In [1]:
import pytest
from pygom.model.ode_utils.compiler import CompileCode

## Recycle cached backend if already found

In [2]:
import time

t1 = time.time()
x = CompileCode()
t2 = time.time()

first_backend_time = t2 - t1

t1 = time.time()
y = CompileCode()
t2 = time.time()

subsequent_backend_time = t2 - t1

assert subsequent_backend_time < first_backend_time

## Reject invalid backend name

In [3]:
with pytest.raises(ValueError):
    z = CompileCode('fake_backend')

# Compile methods (vector, matrix or 3D array)

In [4]:
from pygom.model.transition import Transition
from pygom.model.pygom_model import Model

states = ['S', 'I', 'R']
params = ['beta', 'gamma']
derived_params = [('N', 'S+I+R')]

inf = Transition(origin='S', destination='I', rate='beta*S*I/N')
rec = Transition(origin='I', destination='R', rate='gamma*I')

event_list = [inf, rec]

mod = Model(state=states, param=params, derived_param=derived_params, event=event_list)

In [5]:
namespace = mod._ModelSpec.states_and_parameters_list
compiler = CompileCode()

N = 1000
S = 800
I = 100
R = N - (S+I)
beta = 0.3
gamma = 0.25
t = 0.0

states_and_params = [S, I, R, beta, gamma, N, t]

for method in mod._maths_methods:
    name = method.method_name
    expr = getattr(mod, name).get_equation()

    f = compiler.compileExpr(inputSymb=namespace, inputExpr=expr)
    print(name)
    print(f(states_and_params))
    print()


state_change_matrix
[[-1.  0.]
 [ 1. -1.]
 [ 0.  1.]]

ode
[[-24.]
 [ -1.]
 [ 25.]]

event_rate_vector
[[24.]
 [25.]]

ode_jacobian_params
[[ -80.    0.]
 [  80. -100.]
 [   0.  100.]]

ode_jacobian_states
[[-0.03 -0.24  0.  ]
 [ 0.03 -0.01  0.  ]
 [ 0.    0.25  0.  ]]

event_rates_jacobian_params
[[ 80.   0.]
 [  0. 100.]]

event_rates_jacobian_states
[[0.03 0.24 0.  ]
 [0.   0.25 0.  ]]

ode_hessian_params
[[[0. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 0.]]]

ode_hessian_states
[[[ 0.     -0.0003  0.    ]
  [-0.0003  0.      0.    ]
  [ 0.      0.      0.    ]]

 [[ 0.      0.0003  0.    ]
  [ 0.0003  0.      0.    ]
  [ 0.      0.      0.    ]]

 [[ 0.      0.      0.    ]
  [ 0.      0.      0.    ]
  [ 0.      0.      0.    ]]]

event_rates_hessian_params
[[[0. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 0.]]]

event_rates_hessian_states
[[[0.     0.0003 0.    ]
  [0.0003 0.     0.    ]
  [0.     0.     0.    ]]

 [[0.     0.     0.    ]
  [0.     0.     0.    ]
  [0.     0.     

In [13]:
dp = mod._ModelSpec.derived_parameter_expression_dict

for value in dp.values():
    x = compiler.compileExpr(inputSymb=namespace, inputExpr=value)
    print(x(states_and_params))

1000.0


In [18]:
for key in mod._ModelSpec.derived_parameter_expression_dict.keys():
    print(type(key))

<class 'str'>


In [ ]:
class ArgStore:

    """
    A helper class which gets params ready to pass to compiled functions
    """

    def __init__(self, model_spec):
        self._model_spec = model_spec

    def _compile_derived_params(self):
        """
        Compile the expressions for the derived parameters
        """

        derived_params = self._model_spec.derived_parameter_expression_dict
        derived_params_compiled = dict()

        for dp_name, dp_symbolic in derived_params.items():
            derived_params_compiled[dp_name] = self._SC.compileExpr(
                inputSymb=self._model_spec.states_and_parameters_list,
                inputExpr=dp_symbolic,
                backend=None
            )

        self._derived_params_compiled = derived_params_compiled

    def _eval_derived_params(self, state, time):
        """
        Get derived param values.

        A dp may be a function of (state, params, other dps, t) so
        we need to evaluate them in order they were defined (hoping
        that the user defined them in order)
        """

        state_time_param_value_dict = self._get_state_time_param_dict(state, time)

        if self._derived_params_compiled is None:
            self._compile_derived_params()

        # container for evaluated params
        derived_params_value_dict = dict.fromkeys(self._derived_params_compiled, None)

        for dp_name, dp_func in self._derived_params_compiled.items():
            combined_values = {state_time_param_value_dict | derived_params_value_dict}
            derived_params_value_dict[dp_name] = dp_func(*combined_values)

        return derived_params_value_dict

    def _get_state_time_param_dict(self, state:list[float], time:float) -> list[float]:
        """

        Get dict of state, time and param values (lacking derived params)
        
        """
        if state is None or time is None:
            raise InputError("Have to input both state and time")
        elif not self._model_spec._parameter_store.all_values_set:
                raise InputError("Have not set the parameters yet")

        # State dict:
        state_value_dict = dict.fromkeys(self._model_spec.state_dict.keys(), state)

        # Time dict:
        time_dict = {'t': time}

        # Param dict:
        param_value_dict = self._model_spec.parameter_value_dict

        return {state_value_dict | time_dict | param_value_dict}

    def _get_eval_param(self, state, time):

        dp = self._eval_derived_params
        state_param_time = self._get_state_time_param_dict(state, time)

        return {dp | state_param_time}